In [1]:
from datetime import datetime, timedelta
import pandas as pd
import calendar
from IPython.display import display, HTML


In [2]:

def render_clean_standalone_holiday_calendar(csv_file_path, month, year):
    # 1. Load multi-tier spreadsheet data
    try:
        df = pd.read_csv(csv_file_path)
    except FileNotFoundError:
        print(f"Error: Could not find the file '{csv_file_path}'. Please run your scheduling engine first.")
        return
    
    df['Date'] = pd.to_datetime(df['Date'])
    df_filtered = df[(df['Date'].dt.month == month) & (df['Date'].dt.year == year)]
    
    if df_filtered.empty:
        print(f"No schedule data found for Month: {month}, Year: {year}")
        return

    # Automatically harvest unique staff names for the dropdown menu
    unique_staff = set()
    shift_map = {}
    
    # Helper to clean strings and eliminate 'nan' text artifacts
    def clean_cell_value(val):
        if pd.isna(val) or str(val).strip().lower() in ['nan', 'none', '']:
            return "None"
        return str(val).strip()

    for _, row in df_filtered.iterrows():
        day_num = row['Date'].day
        day_of_week_str = str(row['Day Week'] if 'Day Week' in row.index else row['Day of Week'])
        is_holiday = "[HOLIDAY]" in day_of_week_str
        
        shift_map[day_num] = {
            'is_holiday': is_holiday,
            'day_hospital': clean_cell_value(row['Day Shift (12hr)']),
            'night_hospital': clean_cell_value(row['Night Shift (12hr)']),
            'full_clinic': clean_cell_value(row['Full Clinic (8hr)']),
            'half_clinic': clean_cell_value(row['Half Clinic (4hr)'])
        }
        
        # Pull names for the filter dropdown while safely bypassing holiday closed tags and empty slots
        for col in ['Day Shift (12hr)', 'Night Shift (12hr)', 'Full Clinic (8hr)', 'Half Clinic (4hr)']:
            val_clean = clean_cell_value(row[col])
            if val_clean != "None" and "CLOSED" not in val_clean:
                for name in val_clean.split(", "):
                    unique_staff.add(name)

    cal = calendar.Calendar(firstweekday=6) # Sunday start grid
    month_days = cal.monthdayscalendar(year, month)
    month_name = calendar.month_name[month]

    # 2. Build the HTML Dropdown Selector
    dropdown_html = f"""
    <div style="font-family: 'Segoe UI', Arial, sans-serif; margin-bottom: 20px; background-color: #f8fafc; padding: 15px; border-radius: 6px; border: 1px solid #e2e8f0; display: inline-block;">
        <label for="staffFilter" style="font-weight: 600; margin-right: 12px; color: #0f172a; font-size: 14px;">🔎 Filter Calendar by Staff Member:</label>
        <select id="staffFilter" onchange="filterCalendarView(this.value)" style="padding: 8px 16px; border-radius: 4px; border: 1px solid #cbd5e1; background-color: white; color: #0f172a; font-weight: 500; font-size: 14px; outline: none; cursor: pointer; box-shadow: 0 1px 2px rgba(0,0,0,0.05);">
            <option value="ALL">📋 Show Master Schedule (Full Team View)</option>
    """
    for name in sorted(list(unique_staff)):
        dropdown_html += f'<option value="{name}">{name}</option>'
    dropdown_html += "</select></div>"

    # 3. Construct Calendar Grid Layout
    calendar_html = f"""
    <style>
        .cal-table {{ border-collapse: collapse; font-family: 'Segoe UI', Arial, sans-serif; width: 100%; max-width: 1100px; margin-bottom: 20px; box-shadow: 0 5px 15px rgba(0,0,0,0.06); border-radius: 8px; overflow: hidden; }}
        .cal-header {{ background-color: #0f172a; color: white; text-align: center; font-size: 24px; font-weight: bold; padding: 16px; }}
        .cal-day-name {{ background-color: #f1f5f9; color: #475569; text-align: center; font-weight: 600; padding: 12px; width: 14.28%; border: 1px solid #e2e8f0; text-transform: uppercase; font-size: 11px; letter-spacing: 0.5px; }}
        .cal-cell {{ border: 1px solid #e2e8f0; height: 165px; vertical-align: top; padding: 8px; width: 14.28%; background-color: white; transition: background-color 0.2s; }}
        .cal-empty {{ background-color: #f8fafc; border: 1px solid #e2e8f0; }}
        .day-number {{ font-weight: bold; color: #1e293b; font-size: 14px; margin-bottom: 6px; border-bottom: 1px solid #f1f5f9; padding-bottom: 4px; display: flex; justify-content: space-between; }}
        .holiday-badge {{ font-size: 10px; color: #ef4444; background-color: #fee2e2; padding: 1px 6px; border-radius: 10px; font-weight: 600; letter-spacing: 0.3px; }}
        
        /* 4-Tier Interactive Styling Modules */
        .shift-tag {{ font-size: 11px; padding: 4px 6px; margin: 4px 0; border-radius: 4px; line-height: 1.3; font-weight: 600; display: block; transition: all 0.2s ease; }}
        .tier-day {{ background-color: cyan; color: #0369a1; border-left: 4px solid #0284c7; }}
        .tier-night {{ background-color: navy; color: yellow; border-left: 4px solid #8b5cf6; }}
        .tier-full {{ background-color: #dcfce7; color: #166534; border-left: 4px solid #22c55e; }}
        .tier-half {{ background-color: #fef9c3; color: #854d0e; border-left: 4px solid #eab308; }}
        .tier-holiday-closed {{ background-color: #fef2f2; color: #991b1b; border: 1px dashed #fee2e2; font-weight: 500; text-align: center; font-style: italic; padding: 8px 4px; margin-top: 10px; border-radius: 4px; }}
        
        /* Interactive client-side muted state */
        .js-muted {{ background-color: #f1f5f9 !important; color: #94a3b8 !important; border-left: 4px solid #cbd5e1 !important; opacity: 0.20 !important; font-weight: normal !important; }}
    </style>
    
    <table class="cal-table">
        <tr><th colspan="7" class="cal-header">{month_name} {year}</th></tr>
        <tr>
            <td class="cal-day-name">Sun</td><td class="cal-day-name">Mon</td><td class="cal-day-name">Tue</td>
            <td class="cal-day-name">Wed</td><td class="cal-day-name">Thu</td><td class="cal-day-name">Fri</td>
            <td class="cal-day-name">Sat</td>
        </tr>
    """

    for week in month_days:
        calendar_html += "<tr>"
        for day in week:
            if day == 0:
                calendar_html += '<td class="cal-cell cal-empty"></td>'
            else:
                calendar_html += '<td class="cal-cell">'
                
                has_data = day in shift_map
                day_is_holiday = has_data and shift_map[day]['is_holiday']
                
                if day_is_holiday:
                    calendar_html += f'<div class="day-number">{day} <span class="holiday-badge">🎉 HOLIDAY</span></div>'
                else:
                    calendar_html += f'<div class="day-number">{day}</div>'
                
                if has_data:
                    data = shift_map[day]
                    
                    # 1. Day Hospital Shift Layer
                    if data['day_hospital'] != "None":
                        calendar_html += f'<div class="shift-tag tier-day" data-workers="{data["day_hospital"]}">☀️ {data["day_hospital"]}</div>'
                    
                    # 2. Night Hospital Shift Layer
                    if data['night_hospital'] != "None":
                        calendar_html += f'<div class="shift-tag tier-night" data-workers="{data["night_hospital"]}">🌙 {data["night_hospital"]}</div>'
                    
                    # 3. Full Clinic Layer
                    if data['full_clinic'] != "None":
                        if "CLOSED" in data['full_clinic']:
                            calendar_html += f'<div class="tier-holiday-closed">🏥 Clinic Closed</div>'
                        else:
                            for staff in data['full_clinic'].split(", "):
                                calendar_html += f'<div class="shift-tag tier-full" data-workers="{staff}">💼 {staff}</div>'
                    
                    # 4. Half Clinic Layer (Only renders if it exists and clinics are open)
                    if data['half_clinic'] != "None" and "CLOSED" not in data['half_clinic']:
                        for staff in data['half_clinic'].split(", "):
                            calendar_html += f'<div class="shift-tag tier-half" data-workers="{staff}">⏱️ {staff} (Half)</div>'
                            
                calendar_html += '</td>'
        calendar_html += "</tr>"
    calendar_html += "</table>"

    # 4. Embedded JavaScript Engine
    js_script = """
    <script>
    function filterCalendarView(selectedName) {
        var tags = document.getElementsByClassName('shift-tag');
        for (var i = 0; i < tags.length; i++) {
            var tag = tags[i];
            var workersAttr = tag.getAttribute('data-workers') || '';
            
            if (selectedName === 'ALL') {
                tag.classList.remove('js-muted');
            } else {
                var workerList = workersAttr.split(', ');
                if (workerList.includes(selectedName)) {
                    tag.classList.remove('js-muted');
                } else {
                    tag.classList.add('js-muted');
                }
            }
        }
    }
    </script>
    """

    # 1. Name the output file
    output_filename = "Master_Hospital_Schedule.html"
    
    # 2. Add an optional wrapper to center the grid on the page nicely
    full_page_markup = f"""<!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>Hospital Roster & Clinic Schedule</title>
        <style>
            body {{ 
                background-color: #f8fafc; 
                padding: 20px; 
                margin: 0;
            }}
            /* Keep things clean when saving or printing directly to a PDF document */
            @media print {{
                body {{ background-color: white; padding: 0; }}
                .no-print {{ display: none !important; }}
            }}
        </style>
    </head>
    <body>
    
        {dropdown_html + calendar_html + js_script}
    
    </body>
    </html>"""
    
    # 3. Write and save the document to your local machine storage
    with open(output_filename, "w", encoding="utf-8") as file:
        file.write(full_page_markup)
    
    print(f"🎉 Success! Static file saved to your workspace as: '{output_filename}'")

    display(HTML(dropdown_html + calendar_html + js_script))

# --- Run the application layout ---
render_clean_standalone_holiday_calendar("hospital_holiday_40hr_schedule.csv", month=2, year=2027)


🎉 Success! Static file saved to your workspace as: 'Master_Hospital_Schedule.html'
